# Import Library

In [1]:
import re
import pandas as pd
from pathlib import Path

In [2]:
IN_CSV  = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\preprocessing_result\sirah_simple.csv")
OUT_CSV = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\preprocessing_result\sirah_simple_clean.csv")

DROP_UNKNOWN_BAB = True
DROP_BIBLIO      = True
DROP_EMPTY       = True
DROP_TOO_SHORT   = False
MIN_WORDS        = 5

# Regex Pattern

In [3]:
# ── Regex patterns ───────────────────────────────────────────────────────────
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+|(?<=:)\s+|(?<=;)\s+")
ZWS_RE      = re.compile(r"[\u200b\u200c\u200d\uFEFF]")

# Whitelist singkatan yang valid di teks Sirah
TOKEN_WHITELIST = {
    "SAW", "SWT", "RA", "AS", "QS", "HR", "SM", "AN",
    "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X",
    "DI", "KE", "YA", "LA", "AL", "BI", "WA", "MA", "IN",
    "DAN", "INI", "ITU", "ADA", "HAL",
}

# Kata pendek Indonesia yang sering muncul (2-3 huruf)
COMMON_SHORT_WORDS = {
    "di", "ke", "ya", "la", "mu", "ku", "se", "si", "bi",
    "al", "wa", "ma", "in", "an", "da", "ba", "ha", "ka",
    "dan", "ini", "itu", "ada", "hal", "lah", "pun", "pun",
    "dia", "dua", "apa", "tak", "jua", "bin", "abu", "bab",
    "air", "itu", "saa", "itu", "rak", "sah", "sya", "nya",
    "lah", "jam", "san", "raj", "dai", "akh", "ala", "ber",
}

# Utility Function

## Normalization and Cleanup

In [4]:
def normalize_ws(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\t", " ").replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def light_cleanup(text: str) -> str:
    """Cleaning ringan: hapus karakter non-printable dan simbol bullet."""
    if not isinstance(text, str):
        return ""
    t = "".join(ch for ch in text if ch.isprintable())
    t = t.replace("@", " ").replace("*", " ")
    t = t.replace("\u2022", " ").replace("\u00b7", " ").replace("\u25cf", " ").replace("\u25aa", " ")
    return normalize_ws(t)


def split_sentences(text: str):
    t = normalize_ws(text)
    if not t:
        return []
    return [s.strip() for s in _SENT_SPLIT.split(t) if s.strip()]

## Detection OCR Noise

In [5]:
def token_is_weird(t: str) -> bool:
    """
    Deteksi token noise OCR:
    - Angka berdiri sendiri (8, 3)
    - Huruf besar tunggal bukan di whitelist (W, B, L)
    - Campuran lowercase->UPPERCASE (pHI, wJI)
    - Token uppercase pendek <=3 huruf bukan di whitelist (EL, TJI, KK, GI)
    - Campuran digit + huruf (n1, 8s, t1)
    - Token huruf pendek (<=3 huruf) yang BUKAN kata umum Indonesia (Ug, fts, fs)
    - Token diawali tanda baca (:ebaI)
    """
    t = t.strip()
    if not t:
        return False

    # Bersihkan trailing punctuation untuk pengecekan
    t_clean = re.sub(r"[,.:;!?\-\)\(]+$", "", t)
    t_clean = re.sub(r"^[,.:;!?\-\)\(]+", "", t_clean)

    if not t_clean:
        return True  # token hanya berisi punctuation

    up = t_clean.upper()

    # Token yang memang valid (singkatan agama, angka Romawi)
    if up in TOKEN_WHITELIST:
        return False

    # Token angka murni -> khas noise OCR dalam run gibberish
    if t_clean.isdigit():
        return True

    # Single-letter uppercase (W, B, L, K) -> sangat khas noise OCR
    if len(t_clean) == 1 and t_clean.isalpha() and t_clean == t_clean.upper():
        return True

    # Campuran lowercase->UPPERCASE (pHI, wJI) -> khas noise OCR
    if re.search(r"[a-z][A-Z]{1,}", t_clean):
        return True

    # Token uppercase pendek <=3 huruf, bukan di whitelist (EL, TJI, KK, GI)
    if t_clean.isalpha() and t_clean == t_clean.upper() and len(t_clean) <= 3:
        return True

    # Campuran digit + huruf (A1, 1A, t1, 8s) -> noise
    if re.search(r"\d", t_clean) and re.search(r"[A-Za-z]", t_clean):
        return True

    # Token 1 huruf lowercase (berdiri sendiri) -> kemungkinan besar noise
    if len(t_clean) == 1 and t_clean.isalpha() and t_clean == t_clean.lower():
        return True

    # Token 2-3 huruf yang bukan kata umum Indonesia (Ug, fts, fs, fn, etc)
    if t_clean.isalpha() and 2 <= len(t_clean) <= 3:
        if t_clean.lower() not in COMMON_SHORT_WORDS:
            return True

    # Token diawali tanda baca lalu huruf (:ebaI, .abc) -> noise
    if re.match(r"^[.:;,\-\)\(]{1,2}[A-Za-z]", t):
        return True

    return False

## Remove Gibberish

In [6]:
def purge_isolated_weird_tokens(text: str) -> str:
    """Hapus token individual yang PASTI noise OCR (mix huruf+digit seperti K1Aq)."""
    if not isinstance(text, str):
        return ""
    toks = text.split()
    kept = []
    for t in toks:
        t_clean = re.sub(r"[,.:;!?\-\)\(]+$", "", t)
        t_clean = re.sub(r"^[,.:;!?\-\)\(]+", "", t_clean)
        # Skip if mixed digit+letter (ALWAYS noise in this domain)
        if (t_clean and re.search(r"\d", t_clean) and re.search(r"[A-Za-z]", t_clean)
                and t_clean.upper() not in TOKEN_WHITELIST):
            continue
        kept.append(t)
    return normalize_ws(" ".join(kept))


def remove_gibberish_runs(text: str, min_run_tokens: int = 4) -> str:
    """
    Hapus segmen berupa run token aneh beruntun (>=min_run_tokens token).
    Threshold dari 6 -> 4 agar bisa menangkap "Ug 8 B L fts s 8 GI, fs".
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    s = text
    tokens = list(re.finditer(r"\S+", s))
    if not tokens:
        return normalize_ws(s)

    spans_to_remove = []
    run_start = None
    run_len = 0
    run_end = 0

    for m in tokens:
        tok = m.group(0)
        weird = token_is_weird(tok)

        if weird:
            if run_start is None:
                run_start = m.start()
                run_len = 1
            else:
                run_len += 1
            run_end = m.end()
        else:
            if run_start is not None and run_len >= min_run_tokens:
                spans_to_remove.append((run_start, run_end))
            run_start = None
            run_len = 0

    # finalize last run
    if run_start is not None and run_len >= min_run_tokens:
        spans_to_remove.append((run_start, run_end))

    if not spans_to_remove:
        return normalize_ws(s)

    # remove from back to front (biar index aman)
    out = s
    for a, b in reversed(spans_to_remove):
        out = out[:a] + " " + out[b:]

    return normalize_ws(out)

In [7]:
def is_gibberish_sentence(s: str) -> bool:
    """
    Deteksi kalimat yang mayoritas token-nya aneh.
    Threshold diturunkan ke 50%.
    """
    s = normalize_ws(s)
    if not s:
        return True

    toks = re.findall(r"\S+", s)

    if len(toks) >= 5:
        weird_cnt = sum(token_is_weird(t) for t in toks)
        if weird_cnt / len(toks) >= 0.50:
            return True

    # Simbol aneh berderet
    if re.search(r"[^\w\s]{8,}", s):
        return True
    if re.search(r"[\\{}|<>\[\]~^]{2,}", s):
        return True

    return False


def remove_gibberish(text: str):
    """
    1) Hapus run token aneh (segment-level)
    2) Split kalimat lalu buang kalimat yang gibberish (sentence-level)
    """
    if not isinstance(text, str):
        return "", 0

    # Hapus invisible chars
    t0 = ZWS_RE.sub("", text)

    # (0) hapus token individual yang pasti noise (mix huruf+digit)
    t0 = purge_isolated_weird_tokens(t0)

    # (1) segment-level removal
    t0 = remove_gibberish_runs(t0, min_run_tokens=3)

    # (2) sentence-level removal
    sents = split_sentences(t0)
    if not sents:
        return normalize_ws(t0), 0

    kept = []
    dropped = 0
    for s in sents:
        if is_gibberish_sentence(s):
            dropped += 1
        else:
            kept.append(s)

    return normalize_ws(" ".join(kept)), dropped

# Read Dataset

In [8]:
df = pd.read_csv(IN_CSV, sep=";", encoding="utf-8-sig")
print(f"Rows awal: {len(df)}")
df.head()

Rows awal: 391


,judul_bab,judul_sub_bab,halaman,teks
0,UNKNOWN BAB,UNLABELED SECTION,"1-2, 4-6, 8-33",JUARA1 hAIAl Kata Pengantar Syaikh Muhammad Al...
1,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
2,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
3,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...
4,KEKUASAAN DAN IMARAH DI KALANGAN BANGSA ARAB,UNLABELED SECTION,43,Selagi kita hendak membicarakan masalah kekuas...


# Remove Bibliography and Reference

In [9]:
bab_norm = df["judul_bab"].fillna("").astype(str).str.strip().str.lower()
sub_norm = df["judul_sub_bab"].fillna("").astype(str).str.strip().str.lower()

mask_drop = pd.Series(False, index=df.index)

if DROP_UNKNOWN_BAB:
    mask_drop |= bab_norm.eq("unknown bab")

if DROP_BIBLIO:
    pat_biblio = r"(bibliografi|daftar\s+pustaka|bibliography|references|referensi)"
    mask_drop |= (
        bab_norm.str.contains(pat_biblio, regex=True, na=False) |
        sub_norm.str.contains(pat_biblio, regex=True, na=False)
    )

df = df[~mask_drop].copy()
print(f"Rows setelah filter UNKNOWN/BIBLIO: {len(df)}")

Rows setelah filter UNKNOWN/BIBLIO: 389


C:\Users\Owner\AppData\Local\Temp\ipykernel_5796\3040596085.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bab_norm.str.contains(pat_biblio, regex=True, na=False) |
C:\Users\Owner\AppData\Local\Temp\ipykernel_5796\3040596085.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sub_norm.str.contains(pat_biblio, regex=True, na=False)


# Cleanup

In [10]:
df["teks"] = df["teks"].fillna("").astype(str)
df["teks_clean"] = df["teks"].apply(light_cleanup)

print("Contoh hasil light cleanup:")
df[["teks", "teks_clean"]].head(3)

Contoh hasil light cleanup:


,teks,teks_clean
1,Pada hakikatnya istilah Sirah Nabawiyah merupa...,Pada hakikatnya istilah Sirah Nabawiyah merupa...
2,"Menurut bahasa, Arab artinya padang pasir, tan...","Menurut bahasa, Arab artinya padang pasir, tan..."
3,Ditilik dari silsilah keturunan dan cikal-baka...,Ditilik dari silsilah keturunan dan cikal-baka...


In [11]:
tmp = df["teks_clean"].apply(remove_gibberish)
df["teks_clean"] = tmp.apply(lambda x: x[0])
df["gibberish_removed_count"] = tmp.apply(lambda x: x[1])

print(f"Total kalimat gibberish dibuang: {df['gibberish_removed_count'].sum()}")

Total kalimat gibberish dibuang: 2


In [12]:
if DROP_EMPTY:
    before = len(df)
    df = df[df["teks_clean"].str.strip().ne("")].copy()
    print(f"Drop empty: {before} -> {len(df)} rows")

if DROP_TOO_SHORT:
    before = len(df)
    df["_wc"] = df["teks_clean"].apply(lambda x: len(re.findall(r"\S+", x)))
    df = df[df["_wc"] >= MIN_WORDS].copy()
    df.drop(columns=["_wc"], inplace=True)
    print(f"Drop too short: {before} -> {len(df)} rows")

print(f"\nRows akhir: {len(df)}")

Drop empty: 389 -> 389 rows

Rows akhir: 389


In [13]:
# Quick check: cari sisa gibberish di output
check = df["teks_clean"].str.contains("Ug 8 B L fts", na=False)
if check.any():
    print("\u26a0\ufe0f  MASIH ADA gibberish 'Ug 8 B L fts' yang lolos!")
else:
    print("\u2705 Gibberish 'Ug 8 B L fts' berhasil dihapus!")

# Preview hasil akhir
df[["judul_bab", "judul_sub_bab", "halaman", "teks_clean"]].head()

✅ Gibberish 'Ug 8 B L fts' berhasil dihapus!


,judul_bab,judul_sub_bab,halaman,teks_clean
1,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
2,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
3,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...
4,KEKUASAAN DAN IMARAH DI KALANGAN BANGSA ARAB,UNLABELED SECTION,43,Selagi kita hendak membicarakan masalah kekuas...
5,KEKUASAAN DAN IMARAH DI KALANGAN BANGSA ARAB,Raja-raja di Yaman,43-45,Suku terdahulu yang dikenal di Yaman dari kala...


# Output 

In [14]:
df_final = df[["judul_bab", "judul_sub_bab", "halaman", "teks_clean"]].copy()
df_final.to_csv(OUT_CSV, index=False, sep=";", encoding="utf-8-sig")
print(f"Output disimpan ke: {OUT_CSV}")

Output disimpan ke: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\preprocessing_result\sirah_simple_clean.csv
